In [1]:
import sys
sys.path.append("..")
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from src.features import split_xy
from src.models import build_xgb

train = pd.read_csv("../data/processed/train.csv")
X, y = split_xy(train)

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

print("depth  trees  lr     train AUC   validation AUC")
for depth, trees, lr in [(2, 200, 0.1), (3, 200, 0.1), (3, 600, 0.1),
                         (6, 200, 0.1), (6, 600, 0.1), (3, 600, 0.02)]:
    res = cross_validate(build_xgb(trees, depth, lr), X, y, cv = cv,
                         scoring = "roc_auc", return_train_score = True)
    print(f"{depth:<5d}  {trees:<5d}  {lr:<5}  {res['train_score'].mean():.3f}       {res['test_score'].mean():.3f}")

depth  trees  lr     train AUC   validation AUC
2      200    0.1    0.801       0.781
3      200    0.1    0.825       0.780
3      600    0.1    0.869       0.774
6      200    0.1    0.930       0.775
6      600    0.1    0.988       0.762
3      600    0.02   0.810       0.781


More depth and more trees raise train AUC but lower validation AUC, so XGBoost overfits too. Best results came from small trees or a slow learning rate (about 0.78).